# Task 2: Advanced SQL and Python Database Integration

**Project:** Amazon Sale Report — Data Analytics Internship

This notebook demonstrates SQLite database access, SQL querying, pandas integration,
parameterised queries, analytical window functions, views, and query-plan inspection.

The database contains 128,975 cleaned dataset records. Amount-based results are
labelled as recorded sales because the source includes cancelled and other order
statuses. The dataset has no customer identifier, so customer-level retention
cannot be calculated reliably.


## 1. Imports and database connection

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "amazon_sales.db").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_PATH = PROJECT_ROOT / "data" / "amazon_sales.db"
assert DB_PATH.exists(), f"Database not found: {DB_PATH}"

engine = create_engine(f"sqlite:///{DB_PATH.as_posix()}")
print("Database:", DB_PATH)


Database: C:\Users\sarka\Documents\apexplanet-data-analytics\data\amazon_sales.db


## 2. Verify the database and inspect its schema

In [2]:
with sqlite3.connect(DB_PATH) as conn:
    row_count = conn.execute(
        "SELECT COUNT(*) FROM amazon_sales"
    ).fetchone()[0]
    schema = conn.execute(
        "PRAGMA table_info(amazon_sales)"
    ).fetchall()

print("Record count:", row_count)
print("Column names:", [column[1] for column in schema])
assert row_count == 128975


Record count: 128975
Column names: ['index', 'order_id', 'date', 'status', 'fulfilment', 'sales_channel', 'ship_service_level', 'style', 'sku', 'category', 'size', 'asin', 'courier_status', 'qty', 'currency', 'amount', 'ship_city', 'ship_state', 'ship_postal_code', 'ship_country', 'promotion_ids', 'b2b', 'fulfilled_by']


## 3. Basic SQL: SELECT, WHERE, ORDER BY, and LIMIT

In [3]:
basic_df = pd.read_sql_query(
    text("""
        SELECT order_id, date, category, qty, amount, status
        FROM amazon_sales
        WHERE qty > 1
        ORDER BY amount DESC
        LIMIT :row_limit
    """),
    engine,
    params={"row_limit": 10}
)
basic_df


,order_id,date,category,qty,amount,status
0,402-5239558-4281957,2022-04-05,Set,8,5584.0,Shipped
1,403-4106553-1691525,2022-04-23,Set,5,5495.0,Shipped - Delivered to Buyer
2,406-7195963-2374765,2022-05-02,Set,4,3036.0,Shipped
3,404-4712917-7300332,2022-06-06,Set,2,2894.0,Shipped
4,408-9885439-6698733,2022-04-13,Set,2,2864.0,Shipped
5,408-3363121-6123562,2022-04-06,Western Dress,4,2860.0,Shipped - Delivered to Buyer
6,405-6877720-7865168,2022-05-07,kurta,4,2796.0,Shipped - Returned to Seller
7,408-9361250-7029128,2022-05-19,Set,2,2698.0,Shipped
8,402-9141667-1791537,2022-05-10,Set,2,2676.0,Cancelled
9,403-0343365-3075550,2022-06-06,Set,4,2664.0,Shipped


## 4. Filtering with IN

In [4]:
in_df = pd.read_sql_query(
    text("""
        SELECT category, status, COUNT(*) AS record_count
        FROM amazon_sales
        WHERE category IN (:category1, :category2)
        GROUP BY category, status
        ORDER BY category, record_count DESC
    """),
    engine,
    params={"category1": "Set", "category2": "kurta"}
)
in_df


,category,status,record_count
0,Set,Shipped,30665
1,Set,Shipped - Delivered to Buyer,10644
2,Set,Cancelled,7337
3,Set,Shipped - Returned to Seller,766
4,Set,Shipped - Picked Up,406
5,Set,Pending,252
6,Set,Pending - Waiting for Pick Up,108
7,Set,Shipped - Returning to Seller,73
8,Set,Shipped - Out for Delivery,19
9,Set,Shipping,6


## 5. Recorded sales by category

In [5]:
category_df = pd.read_sql_query(
    text("""
        SELECT category,
               COUNT(*) AS order_records,
               SUM(qty) AS recorded_quantity,
               ROUND(SUM(COALESCE(amount, 0)), 2) AS recorded_sales
        FROM amazon_sales
        GROUP BY category
        ORDER BY recorded_sales DESC
    """),
    engine
)
category_df


,category,order_records,recorded_quantity,recorded_sales
0,Set,50284,45289,39204124.03
1,kurta,49877,45045,21299546.70
2,Western Dress,15500,13943,11216072.69
3,Top,10622,9903,5347792.30
4,Ethnic Dress,1159,1053,791217.66
5,Blouse,926,863,458408.18
6,Bottom,440,398,150667.98
7,Saree,164,152,123933.76
8,Dupatta,3,3,915.00


## 6. Monthly recorded sales trend

In [6]:
monthly_df = pd.read_sql_query(
    text("""
        SELECT strftime('%Y-%m', date) AS month,
               COUNT(*) AS order_records,
               ROUND(SUM(COALESCE(amount, 0)), 2) AS recorded_sales
        FROM amazon_sales
        GROUP BY strftime('%Y-%m', date)
        ORDER BY month
    """),
    engine
)
monthly_df


,month,order_records,recorded_sales
0,2022-03,171,101683.85
1,2022-04,49067,28838708.32
2,2022-05,42040,26226476.75
3,2022-06,37697,23425809.38


## 7. Window functions: moving average and cumulative total

In [7]:
window_df = pd.read_sql_query(
    text("""
        WITH monthly AS (
            SELECT strftime('%Y-%m', date) AS month,
                   SUM(COALESCE(amount, 0)) AS recorded_sales
            FROM amazon_sales
            GROUP BY strftime('%Y-%m', date)
        )
        SELECT month,
               ROUND(recorded_sales, 2) AS recorded_sales,
               ROUND(
                   AVG(recorded_sales) OVER (
                       ORDER BY month
                       ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
                   ), 2
               ) AS moving_average_3_month,
               ROUND(
                   SUM(recorded_sales) OVER (
                       ORDER BY month
                       ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                   ), 2
               ) AS cumulative_recorded_sales
        FROM monthly
        ORDER BY month
    """),
    engine
)
window_df


,month,recorded_sales,moving_average_3_month,cumulative_recorded_sales
0,2022-03,101683.85,101683.85,101683.85
1,2022-04,28838708.32,14470196.09,28940392.17
2,2022-05,26226476.75,18388956.31,55166868.92
3,2022-06,23425809.38,26163664.82,78592678.30


## 8. Query a reusable database view

In [8]:
view_df = pd.read_sql_query(
    "SELECT * FROM vw_category_performance ORDER BY recorded_sales DESC",
    engine
)
view_df


,category,order_records,total_recorded_quantity,recorded_sales,average_amount
0,Set,50284,45289,39204124.03,833.39
1,kurta,49877,45045,21299546.70,455.93
2,Western Dress,15500,13943,11216072.69,762.79
3,Top,10622,9903,5347792.30,526.10
4,Ethnic Dress,1159,1053,791217.66,723.90
5,Blouse,926,863,458408.18,520.33
6,Bottom,440,398,150667.98,358.73
7,Saree,164,152,123933.76,799.57
8,Dupatta,3,3,915.00,305.00


## 9. Parameterised queries and SQL injection safety

In [9]:
category_parameter = "Set"

safe_df = pd.read_sql_query(
    text("""
        SELECT category, COUNT(*) AS record_count
        FROM amazon_sales
        WHERE category = :category
        GROUP BY category
    """),
    engine,
    params={"category": category_parameter}
)
safe_df


,category,record_count
0,Set,50284


In [10]:
# This input is passed as a value, not concatenated into SQL.
injection_like_input = "' OR 1=1 --"

safe_test_df = pd.read_sql_query(
    text("SELECT order_id FROM amazon_sales WHERE category = :category"),
    engine,
    params={"category": injection_like_input}
)

print("Rows returned:", len(safe_test_df))
assert len(safe_test_df) == 0
print("PASS: parameterised input did not bypass the filter.")


Rows returned: 0
PASS: parameterised input did not bypass the filter.


## 10. Query optimisation with EXPLAIN QUERY PLAN

In [11]:
plans = {
    "Date filter": """
        SELECT order_id, date, amount
        FROM amazon_sales
        WHERE date = '2022-04-30'
    """,
    "Category filter": """
        SELECT category, COUNT(*)
        FROM amazon_sales
        WHERE category = 'Set'
        GROUP BY category
    """,
    "Status filter": """
        SELECT order_id, status
        FROM amazon_sales
        WHERE status = 'Cancelled'
    """,
    "SKU lookup": """
        SELECT sku, category, amount
        FROM amazon_sales
        WHERE sku = 'SET389-KR-NP-S'
    """
}

with sqlite3.connect(DB_PATH) as conn:
    for label, query in plans.items():
        print(f"\\n--- {label} ---")
        for plan_row in conn.execute("EXPLAIN QUERY PLAN " + query):
            print(plan_row[-1])


\n--- Date filter ---
SEARCH amazon_sales USING INDEX idx_amazon_sales_date (date=?)
\n--- Category filter ---
SEARCH amazon_sales USING COVERING INDEX idx_amazon_sales_category (category=?)
\n--- Status filter ---
SEARCH amazon_sales USING INDEX idx_amazon_sales_status (status=?)
\n--- SKU lookup ---
SEARCH amazon_sales USING INDEX idx_amazon_sales_sku (sku=?)


## 11. JOIN demonstration using monthly aggregate results

In [12]:
join_df = pd.read_sql_query(
    text("""
        WITH monthly_sales AS (
            SELECT strftime('%Y-%m', date) AS month,
                   ROUND(SUM(COALESCE(amount, 0)), 2) AS recorded_sales
            FROM amazon_sales
            GROUP BY strftime('%Y-%m', date)
        ),
        monthly_orders AS (
            SELECT strftime('%Y-%m', date) AS month,
                   COUNT(*) AS order_records
            FROM amazon_sales
            GROUP BY strftime('%Y-%m', date)
        )
        SELECT s.month, o.order_records, s.recorded_sales
        FROM monthly_sales AS s
        INNER JOIN monthly_orders AS o ON s.month = o.month
        ORDER BY s.month
    """),
    engine
)
join_df


,month,order_records,recorded_sales
0,2022-03,171,101683.85
1,2022-04,49067,28838708.32
2,2022-05,42040,26226476.75
3,2022-06,37697,23425809.38


## 12. Data limitations and conclusions

### Findings and limitations

- The database contains 128,975 records.
- SQL queries demonstrate filtering, sorting, grouping, aggregation, `IN`, `JOIN`,
  views, and window functions.
- Parameterised queries were used instead of string concatenation for user-supplied
  values.
- Index usage was inspected using SQLite's `EXPLAIN QUERY PLAN`.
- The source dataset has an order identifier but no customer identifier. Therefore,
  unique customers, customer retention, and customer-level revenue rankings cannot
  be reliably determined from these fields.
- Recorded amount totals include multiple order statuses unless a query explicitly
  filters them. Missing amounts are treated as zero only for sum calculations;
  this does not recover the missing values.
- The data spans only a limited set of months, so long-term trends should not be
  inferred.

**Conclusion:** SQLite and Python are integrated using SQLAlchemy, pandas, and
sqlite3. The notebook provides a reproducible workflow for Task 2 analysis.


In [13]:
engine.dispose()
print("Notebook analysis completed.")


Notebook analysis completed.
